# ActiveRecord — Dry Run & Rollback via `class_eval`

**Pattern**: inject `before_save` / `before_destroy` callbacks into `ActiveRecord::Base` from outside.

- **Phase 1 (dry run)**: callbacks capture a before-snapshot and `throw :abort` — the DB is never written.
- **Phase 2 (real run)**: callbacks are guarded off — save proceeds normally.
- **Phase 3 (rollback)**: snapshot is used to restore the record via `update_columns`.

## Setup — boot ActiveRecord with a file-based SQLite database

> **Reset DB only:** re-run this cell to drop and re-seed the table (`force: :cascade`).
> **Full reset:** restart the kernel and re-run all cells — re-running only the patch cell adds duplicate callbacks.

In [1]:
require "active_record"

Object.send(:remove_const, :DB_PATH) if defined?(DB_PATH)
DB_PATH = "/lab/ruby/notebooks/ar_demo.db"

ActiveRecord::Base.establish_connection(adapter: "sqlite3", database: DB_PATH)
ActiveRecord::Schema.verbose = false
ActiveRecord::Schema.define do
  create_table :users, force: :cascade do |t|
    t.string  :name
    t.string  :email
    t.integer :age
    t.timestamps
  end
end

class User < ActiveRecord::Base
  self.table_name = "users"
end

User.create!(name: "Alice", email: "alice@example.com", age: 30)
User.create!(name: "Bob",   email: "bob@example.com",   age: 25)

User.all.each { |u| puts "  #{u.name}: age=#{u.age}" }

  Alice: age=30
  Bob: age=25


[#<#<Class:0x00007f4895feaeb8>::User id: 1, name: "Alice", email: "alice@example.com", age: 30, created_at: "2026-06-03 09:51:30.561024000 +0000", updated_at: "2026-06-03 09:51:30.561024000 +0000">, #<#<Class:0x00007f4895feaeb8>::User id: 2, name: "Bob", email: "bob@example.com", age: 25, created_at: "2026-06-03 09:51:30.562513000 +0000", updated_at: "2026-06-03 09:51:30.562513000 +0000">]

## RollbackContext

A plain module that holds the mode flag and the before-snapshot.  
No framework dependency — just a global state object.

In [2]:
module RollbackContext
  @mode      = :real
  @snapshots = []

  def self.dry_run!  = (@mode = :dry;  @snapshots = [])
  def self.real_run! = (@mode = :real)
  def self.dry_run?  = (@mode == :dry)

  def self.add_snapshot(table:, record_id:, operation:, before:)
    @snapshots << { table: table, record_id: record_id, operation: operation, before: before }
  end

  def self.snapshots = @snapshots.dup

  def self.rollback!
    @snapshots.each do |snap|
      next unless snap[:operation] == :update
      klass = ActiveRecord::Base.descendants.find { |k| k.table_name == snap[:table] }
      klass&.find(snap[:record_id])
            &.update_columns(snap[:before].except("id"))
    end
    @snapshots = []
    @mode = :real
  end
end

puts 'RollbackContext ready'

RollbackContext ready


## The patch — `ActiveRecord::Base.class_eval`

Two private methods are injected as `before_save` / `before_destroy` callbacks.  
The `if: -> { RollbackContext.dry_run? }` guard means they are completely skipped during the real run — the patch costs nothing when not in dry-run mode.

In [3]:
# NOTE: run this cell only once per kernel session.
# Re-running it registers duplicate callbacks which distorts the demo output.
# To reset, restart the kernel and re-run all cells from the top.
# The `if:` guard is re-evaluated on every save, so toggling RollbackContext
# mode mid-session takes effect immediately without reloading the patch.
ActiveRecord::Base.class_eval do
  before_save    :capture_before_save,    if: -> { RollbackContext.dry_run? }
  before_destroy :capture_before_destroy, if: -> { RollbackContext.dry_run? }

  private

  def capture_before_save
    unless new_record?
      # attribute_was() gives the pre-change DB value for dirty columns;
      # read_attribute() gives the current value for clean ones.
      before = self.class.column_names.to_h do |col|
        [col, attribute_changed?(col) ? attribute_was(col) : read_attribute(col)]
      end

      RollbackContext.add_snapshot(
        table:     self.class.table_name,
        record_id: id,
        operation: :update,
        before:    before
      )
    end

    throw :abort  # always -- creates are blocked in dry-run too, just not recorded
  end

  def capture_before_destroy
    RollbackContext.add_snapshot(
      table:     self.class.table_name,
      record_id: id,
      operation: :destroy,
      before:    attributes.dup
    )
    throw :abort
  end
end

puts "Patch applied -- ActiveRecord::Base.class_eval callbacks registered"

Patch applied -- ActiveRecord::Base.class_eval callbacks registered


## Phase 1 — Dry Run

Set `dry_run!`, attempt the update.  
`update` returns `false` because the `before_save` callback threw `:abort`.  
The DB is untouched and the snapshot is captured.

In [4]:
RollbackContext.dry_run!

alice  = User.find_by(name: 'Alice')
result = alice.update(age: 99)

puts "update returned: #{result.inspect}   (false = aborted by patch)"
puts "DB -> Alice: age=#{User.find_by(name: 'Alice').age}   (unchanged)"
puts "Snapshot: #{RollbackContext.snapshots.first&.slice(:table, :record_id, :operation)}"

update returned: false   (false = aborted by patch)
DB -> Alice: age=30   (unchanged)
Snapshot: {:table=>"users", :record_id=>1, :operation=>:update}


## Phase 2 — Real Run

Switch to `real_run!`.  
The `if:` guard evaluates to `false`, so the callbacks are completely skipped.  
The save proceeds normally.

In [5]:
RollbackContext.real_run!

User.find_by(name: 'Alice').update!(age: 99)

puts "DB -> Alice: age=#{User.find_by(name: 'Alice').age}   (side effect applied)"

DB -> Alice: age=99   (side effect applied)


## Phase 3 — Rollback

`rollback!` iterates the Phase 1 snapshots and calls `update_columns` on each record.  
`update_columns` bypasses all callbacks (including our own patch) and validations.

In [6]:
RollbackContext.rollback!

puts "DB -> Alice: age=#{User.find_by(name: 'Alice').age}   (restored)"

DB -> Alice: age=30   (restored)
